In [1]:
from IPython import display

%pip install pandas

display.clear_output()

In [2]:
import os
import pandas as pd

Check to see if state smoothing improves precision and recall over frame by frame detection. Ideally, state smoothing should lead to classification as open or closed being more accurate to the ground truth.

In [3]:
root = "/home/reuekennedy/Oyster-Video-Monitoring-System-2026/analysis/annotation_logs"

In [4]:
def getIOU(box1, box2):
  box1X1 = max(box1["x1"], 0)
  box1Y1 = max(box1["y1"], 0)
  box2X1 = max(box2["x1"], 0)
  box2Y1 = max(box2["y1"], 0)

  intersectionX1 = max(box1X1, box2X1)
  intersectionY1 = max(box1Y1, box2Y1)
  intersectionX2 = min(box1["x2"], box2["x2"])
  intersectionY2 = min(box1["y2"], box2["y2"])

  if ((intersectionX2 < intersectionX1) or (intersectionY2 < intersectionY1)):
    return 0

  intersectionArea = (intersectionX2 - intersectionX1) * (intersectionY2 - intersectionY1)

  box1Area = (box1["x2"] - box1X1) * (box1["y2"] - box1Y1)

  box2Area = (box2["x2"] - box2X1) * (box2["y2"] - box2Y1)

  unionArea = box1Area + box2Area - intersectionArea

  return intersectionArea / unionArea


In [5]:
true_positives_smooth = 0
true_positives_raw = 0
total_positives = 0
total_oysters = 0
detected_oysters = 0

for i in range(1,19):
    subdir = root + "/video_" + str(i)
    detections = pd.read_csv(subdir + "/oyster_video_" + str(i) + "_log.csv")
    frames = pd.read_csv(subdir + "/video_" + str(i) + "_frames.csv")
    ground_truth = subdir + "/truth"

    for subdirs, dirs, files in os.walk(ground_truth):
        frame_num = 0
        for file in files:
            if (file != ".DS_Store"):
                true_frame = pd.read_csv(ground_truth + "/" + file)
                total_oysters += len(true_frame)

                detected_frame = detections[detections["frame"] == frames.iloc[frame_num, 0]]
                total_positives += len(detected_frame)

                for idx, row in detected_frame.iterrows():
                    #use iou to find closest match
                    box1X1 = float(detected_frame.at[idx, "x1"])
                    box1Y1 = float(detected_frame.at[idx, "y1"])
                    box1X2 = float(detected_frame.at[idx, "x2"])
                    box1Y2 = float(detected_frame.at[idx, "y2"])
                    box1 = {"y1":box1Y1, "x1":box1X1, "y2":box1Y2, "x2":box1X2}
                    bestIOU = 0
                    match = -1

                    for idx2, row2 in true_frame.iterrows():
                        box2X1 = (float(true_frame.at[idx2, "xcenter"]) - true_frame.at[idx2, "width"] / 2) * 640
                        box2Y1 = (float(true_frame.at[idx2, "ycenter"]) - true_frame.at[idx2, "height"] / 2) * 640
                        box2X2 = (float(true_frame.at[idx2, "xcenter"]) + true_frame.at[idx2, "width"] / 2) * 640
                        box2Y2 = (float(true_frame.at[idx2, "ycenter"]) + true_frame.at[idx2, "height"] / 2) * 640
                        box2 = {"y1":box2Y1, "x1":box2X1, "y2":box2Y2, "x2":box2X2}
                        iou = getIOU(box1, box2)

                        if iou > bestIOU:
                            bestIOU = iou
                            match = idx2

                    #check if detection is a true positive
                    if bestIOU >= 0.3 and match != -1:
                        detected_oysters += 1
                        if ((detected_frame.at[idx, "smoothedLabel"] == "Oyster-Closed" and true_frame.at[match, "class"] == 0) or
                           (detected_frame.at[idx, "smoothedLabel"] == "Oyster-Open" and true_frame.at[match, "class"] == 1)):
                            true_positives_smooth += 1
                        elif ((detected_frame.at[idx, "rawLabel"] == "Oyster-Closed" and true_frame.at[match, "class"] == 0) or
                              (detected_frame.at[idx, "rawLabel"] == "Oyster-Open" and true_frame.at[match, "class"] == 1)):
                            true_positives_raw += 1
                frame_num += 1

print(f"detected oysters: {detected_oysters}\n")
print(f"true positives smooth: {true_positives_smooth}")
print(f"correctly classified: {true_positives_smooth/detected_oysters}")
print(f"precision smooth: {true_positives_smooth/total_positives}")
print(f"recall smooth: {true_positives_smooth/total_oysters}\n")
print(f"true positives raw: {true_positives_raw}")
print(f"correctly classified: {true_positives_raw/detected_oysters}")
print(f"precision raw: {true_positives_raw/total_positives}")
print(f"recall raw: {true_positives_raw/total_oysters}")

FileNotFoundError: [Errno 2] No such file or directory: '/home/reuekennedy/Oyster-Video-Monitoring-System-2026/analysis/annotation_logs/video_1/oyster_video_1_log.csv'

Smoothed class label appear to more frequently match the ground truth than raw, frame-only labels. 

Note: Performance will be worse than model testing metrics since detection is only done once every 10 frames in the browser, leading to boxes not always aligning well with ground truth annotations. Additionally, ground truth annotations were done at 30 frames per second while videos are run at much higher fps in browser.